# Project: RAG - Q&A on Private Documents using LangChain, OpenAI and Pinecone



In [16]:
pip install -q -r ./requirements.txt # This notebook uses the versions of the libraries (OpenAI, LangChain, etc.) shown in requirements.txt.

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

### Loading Documents

In [2]:
# loading PDF, DOCX and TXT files as LangChain Documents
def load_document(file):
    import os
    name, extension = os.path.splitext(file)

    if extension == '.pdf':
        # Moved from langchain.document_loaders to langchain_community.document_loaders (v0.1.0+)
        from langchain_community.document_loaders import PyPDFLoader
        print(f'Loading {file}')
        loader = PyPDFLoader(file)
    elif extension == '.docx':
        # Moved from langchain.document_loaders to langchain_community.document_loaders (v0.1.0+)
        from langchain_community.document_loaders import Docx2txtLoader
        print(f'Loading {file}')
        loader = Docx2txtLoader(file)
    elif extension == '.txt':
        # Moved from langchain.document_loaders to langchain_community.document_loaders (v0.1.0+)
        from langchain_community.document_loaders import TextLoader
        loader = TextLoader(file)
    else:
        print('Document format is not supported!')
        return None

    data = loader.load()
    return data


In [3]:
# wikipedia
def load_from_wikipedia(query, lang='en', load_max_docs=2):
    # Moved from langchain.document_loaders to langchain_community.document_loaders (v0.1.0+)
    from langchain_community.document_loaders import WikipediaLoader
    loader = WikipediaLoader(query=query, lang=lang, load_max_docs=load_max_docs)
    data = loader.load()
    return data


### Chunking Data

In [4]:
def chunk_data(data, chunk_size=256):
    # Moved from langchain.text_splitter to langchain_text_splitters (v0.2.0+)
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=0)
    chunks = text_splitter.split_documents(data)
    return chunks


### Calculating Cost

In [5]:
def print_embedding_cost(texts):
    import tiktoken
    enc = tiktoken.encoding_for_model('text-embedding-3-small')
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    # check prices here: https://openai.com/pricing
    print(f'Total Tokens: {total_tokens}')
    print(f'Embedding Cost in USD: {total_tokens / 1000 * 0.00002:.6f}')

### Embedding and Uploading to a Vector Database (Pinecone)

In [6]:
def insert_or_fetch_embeddings(index_name, chunks):
    import pinecone
    # langchain_community.vectorstores.Pinecone deprecated and incompatible with pinecone v7+
    # Use langchain_pinecone.PineconeVectorStore instead
    from langchain_pinecone import PineconeVectorStore
    from langchain_openai import OpenAIEmbeddings
    from pinecone import ServerlessSpec

    pc = pinecone.Pinecone()

    embeddings = OpenAIEmbeddings(model='text-embedding-3-small', dimensions=1536)  # 512 works as well

    # loading from existing index
    if index_name in pc.list_indexes().names():
        print(f'Index {index_name} already exists. Loading embeddings ... ', end='')
        vector_store = PineconeVectorStore.from_existing_index(index_name, embeddings)
        print('Ok')
    else:
        # creating the index and embedding the chunks into the index
        print(f'Creating index {index_name} and embeddings ...', end='')

        # creating a new index
        pc.create_index(
            name=index_name,
            dimension=1536,
            metric='cosine',
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )

        # processing the input documents, generating embeddings using the provided `OpenAIEmbeddings` instance,
        # inserting the embeddings into the index and returning a new PineconeVectorStore object.
        vector_store = PineconeVectorStore.from_documents(chunks, embeddings, index_name=index_name)
        print('Ok')

    return vector_store


In [7]:
def delete_pinecone_index(index_name='all'):
    import pinecone
    pc = pinecone.Pinecone()
    
    if index_name == 'all':
        indexes = pc.list_indexes().names()
        print('Deleting all indexes ... ')
        for index in indexes:
            pc.delete_index(index)
        print('Ok')
    else:
        print(f'Deleting index {index_name} ...', end='')
        pc.delete_index(index_name)
        print('Ok')
    

### Asking and Getting Answers

In [8]:
def ask_and_get_answer(vector_store, q, k=3):
    # RetrievalQA removed in langchain v1.0+; replaced with LCEL chain
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.runnables import RunnablePassthrough
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model='gpt-5.2', temperature=1)
    retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': k})

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    prompt = ChatPromptTemplate.from_template(
        "Answer the question based only on the following context:\n\n{context}\n\nQuestion: {question}"
    )

    # LCEL chain: retrieve context → format → prompt → LLM → parse output
    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    answer = chain.invoke(q)
    return {"query": q, "result": answer}


### Running Code

In [9]:
# import warnings
# warnings.filterwarnings('ignore')

#### Ask a PDF

In [10]:
data = load_document('files/us_constitution.pdf')
# print(data[1].page_content)
# print(data[10].metadata)

print(f'You have {len(data)} pages in your data')
print(f'There are {len(data[20].page_content)} characters in the page')

Loading files/us_constitution.pdf
You have 41 pages in your data
There are 963 characters in the page


In [11]:
# data = load_document('files/the_great_gatsby.docx')
# print(data[0].page_content)

In [12]:
# data = load_from_wikipedia('GPT-5.4', 'de')
# print(data[0].page_content)

In [11]:
chunks = chunk_data(data)
print(len(chunks))
# print(chunks[10].page_content)

177


In [12]:
print_embedding_cost(chunks)

Total Tokens: 10634
Embedding Cost in USD: 0.000213


In [13]:
delete_pinecone_index()

Deleting all indexes ... 
Ok


In [14]:
index_name = 'askadocument'
vector_store = insert_or_fetch_embeddings(index_name=index_name, chunks=chunks)

Creating index askadocument and embeddings ...Ok


In [15]:
q = 'What is the Bill of Rights?'
answer = ask_and_get_answer(vector_store, q)
print(answer)

{'query': 'What is the Bill of Rights?', 'result': 'The Bill of Rights is the first ten amendments to the United States Constitution, which enumerate certain rights retained by the people and restrict the power of the government.'}


#### While Loop for Asking Questions

In [18]:
import time
i = 1
print('Write Quit or Exit to quit.')
while True:
    q = input(f'Question #{i}: ')
    i = i + 1
    if q.lower() in ['quit', 'exit']:
        print('Quitting ... bye bye!')
        time.sleep(2)
        break
    
    answer = ask_and_get_answer(vector_store, q)
    print(f'\nAnswer: {answer}')
    print(f'\n {"-" * 50} \n')

    

Write Quit or Exit to quit.

Answer: {'query': 'What is the first amendment?', 'result': 'The first amendment is a part of the United States Constitution that protects the freedom of religion, speech, press, assembly, and the right to petition the government.'}

 -------------------------------------------------- 

Quitting ... bye bye!


#### Ask Wikipedia

In [19]:
delete_pinecone_index()

Deleting all indexes ... 
Ok


In [20]:
data = load_from_wikipedia('Google Gemini', 'de')
chunks = chunk_data(data)

In [21]:
chunks

[Document(metadata={'title': 'Google Gemini', 'summary': 'Google Gemini (ehemals Google Bard) ist ein von Google entwickelter KI-basierter, multimodaler Chatbot. Er wurde als direkte Reaktion auf den Erfolg von ChatGPT entwickelt und im März 2023 zunächst unter dem Namen Bard veröffentlicht. Nach umfangreichen Updates und einem Rebranding ist Gemini mittlerweile das zentrale KI-Produkt von Google und in über 230 Ländern sowie mehr als 70 Sprachen verfügbar.\n\n', 'source': 'https://de.wikipedia.org/wiki/Google_Gemini'}, page_content='Google Gemini (ehemals Google Bard) ist ein von Google entwickelter KI-basierter, multimodaler Chatbot. Er wurde als direkte Reaktion auf den Erfolg von ChatGPT entwickelt und im März 2023 zunächst unter dem Namen Bard veröffentlicht. Nach umfangreichen'),
 Document(metadata={'title': 'Google Gemini', 'summary': 'Google Gemini (ehemals Google Bard) ist ein von Google entwickelter KI-basierter, multimodaler Chatbot. Er wurde als direkte Reaktion auf den Erf

In [22]:
index_name = 'gemini'
vector_store = insert_or_fetch_embeddings(index_name=index_name, chunks=chunks)

Creating index gemini and embeddings ...Ok


In [23]:
q = 'Was ist Google Gemini?'
answer = ask_and_get_answer(vector_store, q)
print(answer)

{'query': 'Was ist Google Gemini?', 'result': 'Google Gemini ist der Name des Google-Chatbots, der sowohl kostenlos als auch kostenpflichtig mit erweitertem Zugang zu Updates und Funktionen verfügbar ist. Es ist das zentrale KI-Produkt von Google, das in über 230 Ländern und mehr als 70 Sprachen verfügbar ist. Der Name Gemini steht für die Partnerschaft der beiden großen KI-Labore von Google und ist eine Anspielung auf das NASA-Projekt Gemini.'}
